In [1]:
import time
import clifford as cl
import numpy as np

### 1.1 Initialization of the Algebraic Space
To represent $n$ qubits, we first construct a real geometric (Clifford) algebra $\mathcal{C}\ell_{2n, 0}$. The base vector space requires $2n$ dimensions to properly construct $n$ pairs of creation and annihilation operators (the Witt basis) later on. Here, we define the algebra and extract its orthonormal generators $e_i$.

In [2]:
# Define the number of qubits
n = 2

# Initialize the Clifford algebra Cl(2n, 0)
layout, blades = cl.Cl(2*n, 0)

# Extract the orthonormal generators (standard basis vectors e_1 to e_2n)
g = [blades[f'e{i+1}'] for i in range(2*n)]


### 1.2 Construction of the Witt Basis (Creation and Annihilation Operators)
To model quantum states, we transition to a complexified geometric algebra and construct the Witt basis. For $n$ qubits, we define $n$ pairs of mutually null vectors: the annihilation operators $f_k$ and the creation operators $f_k^\dagger$. These are constructed from the orthogonal basis vectors $e_k$ as follows:

$$f_k = \frac{1}{2}(e_k - i e_{k+n}), \quad f_k^\dagger = \frac{1}{2}(e_k + i e_{k+n})$$

In [4]:
# Initialize empty lists for annihilation (f) and creation (f_dag) operators
f = []
f_dag = []

# Construct the Witt basis pairs for each qubit
for i in range(n):
    f.append((1/2) * (g[i] - 1j*g[i+n]))
    f_dag.append((1/2) * (g[i] + 1j*g[i+n]))

### 1.3 Primitive Idempotent (Vacuum State) and Basis States Generation
In the geometric formulation, the quantum vacuum state $|00\dots 0\rangle$ is represented by a primitive idempotent $I$, defined as the geometric product of the pairs $f_k f_k^\dagger$. Its dual (corresponding to the bra vector $\langle 00\dots 0|$) is denoted as $I^\dagger$ and requires reversing the multiplication order.

$$I = \prod_{k=0}^{n-1} f_k f_k^\dagger, \quad I^\dagger = \prod_{k=n-1}^{0} f_k f_k^\dagger$$

The $2^n$ computational basis states are then generated by systematically applying the creation operators $f_k^\dagger$ to the vacuum state $I$. Similarly, the dual basis states are generated by applying the annihilation operators $f_k$ to $I^\dagger$.

In [5]:
# Initialize the primitive idempotent (vacuum state) and its dual
I = 1
I_dag = 1

# Construct the idempotents
for i in range(n):
    I *= f[i] * f_dag[i]
    # The dual idempotent requires reverse ordering
    I_dag *= f[n-i-1] * f_dag[n-i-1]

# Initialize lists to store the 2^n basis states (kets) and their duals (bras)
qubit = []
qubit_dags = []

# Generate all 2^n computational basis states
for i in range(2**n):
    l = 1
    
    # Convert index to a binary string of length n (e.g., 0 -> '00', 1 -> '01')
    bin_i = bin(i)[2:].zfill(n)
    
    # 1. Construct the state (ket) using creation operators
    for idx, bit in enumerate(bin_i):
        if bit == '1':
            l *= f_dag[idx]
    
    q = l * I
    qubit.append(q)
    
    # 2. Construct the dual state (bra) using annihilation operators
    l = 1
    for idx, bit in enumerate(bin_i):
        if bit == '1':
            l *= f[idx]
            
    q_dag = I_dag * l
    qubit_dags.append(q_dag)

### 1.4 Quantum Gates as Geometric Transformations
In standard quantum computing, logic gates are represented by unitary matrices. In our geometric framework, these gates are naturally expressed as elements of the Clifford algebra (multivectors) using the Witt basis operators. 

The single-qubit Pauli gates for the $m$-th qubit are defined as:
$$X_m = f_m^\dagger + f_m$$
$$Y_m = i(f_m^\dagger - f_m)$$
$$Z_m = f_m f_m^\dagger - f_m^\dagger f_m$$

The Hadamard gate is a linear combination of the $X$ and $Z$ operations:
$$H_m = \frac{1}{\sqrt{2}}(X_m + Z_m)$$

For two-qubit operations, the general Controlled-NOT (CNOT) gate acting on a control qubit $c$ and a target qubit $t$ is constructed using geometric projectors on the control qubit and the $X_t$ operator on the target qubit:
$$CNOT_{c, t} = f_c f_c^\dagger - f_c^\dagger f_c (f_t^\dagger + f_t)$$

In [6]:
# --- Pauli Gates ---

def X(m):
    # Pauli-X (Bit-flip) on qubit m
    return f_dag[m] + f[m]

def Y(m):
    # Pauli-Y (Phase and Bit-flip) on qubit m
    return 1j*f_dag[m] - 1j*f[m]

def Z(m):
    # Pauli-Z (Phase-flip) on qubit m
    return f[m]*f_dag[m] - f_dag[m]*f[m]

# --- Identity Operator ---

def id(m):
    # Identity operation on qubit m
    return 1

# --- Controlled-NOT Gate ---

def C_not(c, t):
    # Controlled-NOT gate (Control: qubit c, Target: qubit t)
    # Uses geometric projectors on the control qubit
    return (f[c]*f_dag[c] - f_dag[c]*f[c]*(f_dag[t] + f[t]))

# --- Hadamard Gate ---

def H(m):
    # Hadamard gate on qubit m
    return (1/np.sqrt(2)) * (X(m) + Z(m))

### 1.5 Quantum Measurement and Oracles
In the standard matrix model, measurement causes the wavefunction to collapse, returning a classical bit. In our geometric framework, we simulate measurement on the computational basis using algebraic projection operators. The projector onto the state $|0\rangle$ for the $m$-th qubit is defined as:
$$P_{|0\rangle}^{(m)} = f_m f_m^\dagger$$

If the state vector contains the $|0\rangle$ component, this projector leaves it invariant. If it is in the $|1\rangle$ state, the projector annihilates it to zero.

**The Oracles:**
Deutsch's algorithm determines whether a hidden boolean function $f(x)$ is constant or balanced. We define two representative oracles:
1. **Constant Oracle ($f(x) = 0$):** Represented by the geometric identity, as it leaves the target qubit unchanged.
2. **Balanced Oracle ($f(x) = x$):** Represented by the $CNOT$ gate, which flips the target qubit based on the state of the data qubit.

In [7]:
# --- Quantum Measurement (Projection) ---

def measure(m):
    # Returns the geometric projector onto the |0> state for qubit m
    # Mathematically: f_m * f_m^\dagger
    return f[m] * f_dag[m]

# --- Deutsch's Algorithm Oracles ---

# 1. Constant Oracle: f(x) = 0 
# Leaves the quantum state completely unchanged
U_f_constant = 1

# 2. Balanced Oracle: f(x) = x 
# Flips target (qubit 1) if control (qubit 0) is |1>
U_f_balanced = C_not(0, 1)

### 1.5 The Core Concept of Deutsch's Algorithm & Oracles

Deutsch's algorithm is the fundamental quantum algorithm demonstrating an advantage over classical computing. It solves the following problem:

Given a black-box boolean function (Oracle) $f: \{0, 1\} \rightarrow \{0, 1\}$, determine if the function is:
1. **Constant:** Returns the same output for all inputs ($f(0) = f(1)$).
2. **Balanced:** Returns different outputs for different inputs ($f(0) \neq f(1)$).

A classical computer requires exactly **two** evaluations (queries) to determine the nature of the function. A quantum computer requires only **one**.

**The Quantum Mechanism (Phase Kickback):**
By initializing the data qubit in a superposition state $|+\rangle$ and the target qubit in the $|-\rangle$ state, the oracle evaluates the function for both inputs simultaneously. Instead of simply altering classical bit values, the oracle imprints the function's output onto the phase of the data qubit—a quantum phenomenon known as *phase kickback*:
$$|x\rangle |-\rangle \xrightarrow{U_f} (-1)^{f(x)} |x\rangle |-\rangle$$

A final Hadamard transformation on the data qubit creates interference:
* If the function is **constant**, constructive interference yields the state $|0\rangle$ with 100% probability.
* If the function is **balanced**, the opposing phases cause destructive interference for $|0\rangle$, yielding the state $|1\rangle$ with 100% probability.

**Our Geometric Implementation:**
In our Complex Geometric Algebra framework, we define two representative oracles:
* **Constant Oracle ($f(x) = 0$):** Represented mathematically by the geometric identity ($1$), as it leaves the quantum state completely unchanged.
* **Balanced Oracle ($f(x) = x$):** Represented by the $CNOT$ multivector.

Measurement is then simulated using the algebraic projector $P_{|0\rangle}^{(0)} = f_0 f_0^\dagger$.

### 1.6 Execution of Deutsch's Algorithm in GA
Here we execute the full quantum circuit using our geometric multivectors. The sequence of operations is exactly equivalent to the standard quantum circuit:
1. Initialize the system in the $|00\rangle$ state.
2. Prepare the data qubit in $|+\rangle$ and the target qubit in $|-\rangle$ using $X$ and $H$ operations.
3. Apply the chosen geometric Oracle.
4. Apply the final Hadamard transformation to the data qubit.
5. Project (measure) the data qubit to determine the nature of the function.

In [9]:
def deutsch_algorithm_GA(oracle, oracle_name):
    print(f"--- Running algorithm for Oracle: {oracle_name} ---")
    
    # 1. Initialization to state |00>
    state = qubit[0]
    
    # 2. Prepare target qubit in state |1> -> total state |01>
    state = X(1) * state
    
    # 3. Superposition (Hadamard on both qubits)
    state = H(0) * H(1) * state
    
    # 4. Apply the chosen oracle
    state = oracle * state
    
    # 5. Final interference (Hadamard on the data qubit)
    state = H(0) * state
    
    # 6. Measurement of the data qubit using the geometric projector
    state_measured = measure(0) * state
    
    # Evaluate the result
    print("MEASUREMENT RESULT:")
    if state == state_measured:
        print("-> Measured state |0>: Function is CONSTANT")
    elif state_measured == 0:
        print("-> Measured state |1>: Function is BALANCED")
        
    # Calculate classical probabilities of the entire system
    p_00 = abs(2**n * (qubit_dags[0] * state).value[0])**2
    p_01 = abs(2**n * (qubit_dags[1] * state).value[0])**2
    p_10 = abs(2**n * (qubit_dags[2] * state).value[0])**2
    p_11 = abs(2**n * (qubit_dags[3] * state).value[0])**2
    
    # Marginalization: Probabilities for the data qubit only
    p_q0_is_0 = p_00 + p_01
    p_q0_is_1 = p_10 + p_11
    
    print("\nProbabilities for the data qubit:")
    print(f"P(|0⟩) = {p_q0_is_0 * 100:.0f} %")
    print(f"P(|1⟩) = {p_q0_is_1 * 100:.0f} %\n")

# --- Execution ---
start = time.perf_counter()

# Test both oracles
deutsch_algorithm_GA(U_f_constant, "Constant (Identity)")
deutsch_algorithm_GA(U_f_balanced, "Balanced (CNOT)")

end = time.perf_counter()
print(f"⏱️ Total execution time in GA: {end - start:.6f} seconds")

--- Running algorithm for Oracle: Constant (Identity) ---
MEASUREMENT RESULT:
-> Measured state |0>: Function is CONSTANT

Probabilities for the data qubit:
P(|0⟩) = 100 %
P(|1⟩) = 0 %

--- Running algorithm for Oracle: Balanced (CNOT) ---
MEASUREMENT RESULT:
-> Measured state |1>: Function is BALANCED

Probabilities for the data qubit:
P(|0⟩) = 0 %
P(|1⟩) = 100 %

⏱️ Total execution time in GA: 0.001345 seconds
